In [11]:
import pandas as pd
import os

In [12]:
# Step 1: Setup
data_dir = r"data\coordinates\yearly_coordinates_pm25"
output_dir = data_dir

years = list(range(2014, 2024))  # 2014 to 2023
dataframes = []

In [13]:
#Read files and append year info
for year in years:
    file_path = os.path.join(data_dir, f"{year}.csv")  # Assumes all files are in the same directory
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        df['Year'] = year  # Add a column for the year
        dataframes.append(df)

#Combine all data
combined_df = pd.concat(dataframes, ignore_index=True)

#Remove duplicates within same year (if any)
combined_df = combined_df.drop_duplicates(subset=['Latitude', 'Longitude', 'Year'])

In [14]:
#Get unique coordinates and assign unique ID
unique_coords = combined_df[['Latitude', 'Longitude']].drop_duplicates().reset_index(drop=True)
unique_coords['UID'] = range(1, len(unique_coords) + 1)

In [15]:
#Merge ID back into combined_df
combined_with_id = combined_df.merge(unique_coords, on=['Latitude', 'Longitude'])

In [16]:
#Group by ID and aggregate year appearances
coordinate_years_df = combined_with_id.groupby(['UID', 'Latitude', 'Longitude'])['Year'] \
    .apply(sorted) \
    .reset_index(name='Years_Present')

In [17]:
#One-hot encoded version
one_hot = pd.get_dummies(combined_with_id['Year'], prefix='Year')
combined_df_one_hot = pd.concat([combined_with_id[['UID', 'Latitude', 'Longitude']], one_hot], axis=1)
combined_df_one_hot = combined_df_one_hot.groupby(['UID', 'Latitude', 'Longitude']).max().reset_index()

In [18]:
# Save to CSV
coordinate_years_df.to_csv(os.path.join(output_dir, 'unique_coordinates_with_years_and_id.csv'), index=False)
combined_df_one_hot.to_csv(os.path.join(output_dir, 'coordinate_year_presence_onehot_with_id.csv'), index=False)